# UT1.5 - Pandas: estructuras, indexación y análisis

**Módulo:** Análisis de Datos con Python

**Resultado de Aprendizaje:** RA1 - Selecciona y utiliza las librerías principales de Python para la importación, manipulación, limpieza y exportación de datos.

**Nombre del Alumno/a:** ________________________________________

**Instrucciones:** Completa los bloques `# TU CÓDIGO AQUÍ` y ejecuta la celda de test correspondiente para comprobar tu solución.

In [ ]:
%reset -f
%run notebook_setup.py
%whos

## 1. Pandas (CE1.a, CE1.d)

Pandas aporta dos estructuras sobre NumPy:

- **`Series`**: array unidimensional **con etiquetas** (el índice).
- **`DataFrame`**: tabla bidimensional con filas y columnas etiquetadas; cada columna es una `Series` que comparte el índice.

Anatomía de un `DataFrame`:

| Parte | Atributo | Contenido |
| :--- | :--- | :--- |
| Filas | `df.index` | Etiquetas de fila (por defecto `RangeIndex(0, 1, 2...)`) |
| Columnas | `df.columns` | Nombres de columna |
| Datos | `df.values` | Array de NumPy con los valores (sin etiquetas) |
| Tipos | `df.dtypes` | Tipo de cada columna |

### 1.1. Series y DataFrame

Una `Series` se crea a partir de una lista o array más un índice opcional. Un `DataFrame` se crea a partir de un diccionario de listas.

In [ ]:
import numpy as np
import pandas as pd

# Series: valores + etiquetas
s = pd.Series([120.5, 340.0, 89.9], index=["Ana", "Pedro", "Elena"], name="gasto")
print(s)
print(f"\nValor de Pedro: {s['Pedro']}")

# DataFrame: diccionario de columnas
df = pd.DataFrame({
    "cliente": ["Ana", "Pedro", "Elena", "Luis"],
    "categoria": ["Hogar", "Tech", "Hogar", "Tech"],
    "gasto": [120.5, 340.0, 89.9, 210.0],
})
print("\n", df)
print("\nÍndice:", df.index.tolist())
print("Columnas:", df.columns.tolist())
print("\nTipos:\n", df.dtypes)

### 1.2. Indexación: `.loc[]`, `.iloc[]` y máscaras booleanas

Pandas tiene dos indexadores explícitos y una forma implícita de seleccionar datos:

| Indexador | Criterio | Extremo final | Ejemplo |
| :--- | :--- | :--- | :--- |
| `.loc[]` | Etiquetas del índice y nombres de columna | **Se incluye** | `df.loc[1:2, ["cliente", "gasto"]]` |
| `.iloc[]` | Posiciones enteras (0, 1, 2...) | **No se incluye** | `df.iloc[1:3, 0:2]` |
| `df[condición]` | Máscara booleana | — | `df[df["gasto"] > 150]` |

Regla mnemotécnica: **loc** → *labels*; **iloc** → *integer location*.

Reglas prácticas:

- Usa siempre `.loc[]` o `.iloc[]` para leer y, sobre todo, para **escribir**.
  `df[df["gasto"] > 0]["gasto"] = 1` puede no modificar el DataFrame original (*chained assignment*);
  lo correcto es `df.loc[df["gasto"] > 0, "gasto"] = 1`.
- Las condiciones combinadas van **cada una entre paréntesis** y con `&` (*and*), `|` (*or*), `~` (*not*);
  nunca con `and`, `or` o `not` de Python.
- `df.loc[condición, "columna"]` devuelve una `Series`; `.values` da el array de NumPy subyacente.
- Para seleccionar una sola columna, `df["columna"]` (Serie) es equivalente a `df.loc[:, "columna"]`.

#### Ejemplo resuelto 1: selección de columnas y filas con `.loc[]`
Con `.loc[]` se selecciona **por etiqueta** y el extremo final **sí** se incluye.

In [ ]:
# Todas las filas de una columna -> Series
print(df.loc[:, "gasto"])

# Varias columnas -> DataFrame
print("\n", df.loc[:, ["cliente", "gasto"]])

# Filas con etiqueta 1 y 2 (ambas incluidas), columnas por nombre
print("\n", df.loc[1:2, ["cliente", "gasto"]])

#### Ejemplo resuelto 2: selección por posición con `.iloc[]`
Con `.iloc[]` se selecciona **por posición entera** y el extremo final **no** se incluye.

In [ ]:
# Primera fila (todas las columnas)
print(df.iloc[0])

# Filas 0 y 1; columnas 0 y 2
print("\n", df.iloc[0:2, [0, 2]])

# Última fila y última columna con índices negativos
print("\nÚltimo gasto:", df.iloc[-1, -1])

#### Ejemplo resuelto 3: filtrado booleano
Una comparación sobre una columna devuelve una `Series` de `True`/`False` con el mismo índice: es una **máscara**.
Al indexar el DataFrame con la máscara se conservan las filas que son `True`.

In [ ]:
mascara = df["gasto"] > 150
print(mascara)

# Solo las filas que cumplen la condición
print("\n", df[mascara])

# Varias condiciones: paréntesis + & (and) / | (or) / ~ (not)
print("\n", df[(df["categoria"] == "Tech") & (df["gasto"] > 200)])
print("\n", df[~(df["categoria"] == "Tech")])

#### Ejemplo resuelto 4: `.loc[]` combinado con máscara, lectura y escritura
Esta es la forma recomendada de leer una columna de las filas filtradas y de modificar valores.

In [ ]:
# Lectura: filas filtradas, solo la columna gasto
print(df.loc[df["gasto"] > 150, "gasto"])

# Escritura: modificar solo las filas que cumplen la condición
df.loc[df["cliente"] == "Ana", "categoria"] = "Premium"
print("\n", df)

#### Ejemplo resuelto 5: el mismo resultado con `.loc[]` y `.iloc[]`
| Objetivo | Con `.loc[]` | Con `.iloc[]` |
| :--- | :--- | :--- |
| Una celda | `df.loc[2, "gasto"]` | `df.iloc[2, 2]` |
| Un rango de filas | `df.loc[1:3]` (4 filas si el índice es 0..4) | `df.iloc[1:3]` (2 filas) |
| Filas y columnas | `df.loc[1:2, "cliente":"gasto"]` | `df.iloc[1:3, 0:3]` |
| Índice no numérico | `df.loc["Ana"]` | no aplica |

> Si el índice no es un `RangeIndex` (por ejemplo, fechas o identificadores), `.iloc[]` sigue funcionando por posición,
> pero `.loc[]` necesita las etiquetas reales.

### 1.3. Ejercicios

Los ejercicios usan `pd` y `np`, importados en el apartado 1.1.

#### Ejercicio 1.1: Transformación e Impuestos (CE1.d)
Implementa `procesar_ventas(df, min_importe)` para:
1. Filtrar las filas con `importe > min_importe`.
2. Calcular la columna `iva` (21 % del importe).
3. Ordenar el resultado por `importe` de forma descendente.

In [ ]:
def procesar_ventas(df: pd.DataFrame, min_importe: float) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función procesar_ventas")

In [ ]:
# TEST 1.1
df_ventas = pd.DataFrame({
    "transaccion_id": [1, 2, 3, 4],
    "importe": [50.0, 150.0, 200.0, 80.0]
})

res_11 = procesar_ventas(df_ventas, min_importe=90.0)
assert list(res_11["transaccion_id"]) == [3, 2]
assert np.isclose(res_11.loc[res_11["transaccion_id"] == 3, "iva"].values[0], 42.0)
print("✅ Test Ejercicio 1.1 superado.")

#### Ejercicio 1.2: Agrupación y Agregación Múltiple (CE1.d)
Implementa `resumen_categoria(df)` para agrupar por `categoria` calculando:
- `total_ventas`: suma de la columna `ventas`.
- `promedio_unidades`: media de la columna `unidades`.

El resultado debe ser un `DataFrame` con `categoria` como columna (no como índice).

In [ ]:
def resumen_categoria(df: pd.DataFrame) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función resumen_categoria")

In [ ]:
# TEST 1.2
df_cat = pd.DataFrame({
    "categoria": ["Tech", "Hogar", "Tech", "Hogar"],
    "ventas": [100.0, 50.0, 300.0, 150.0],
    "unidades": [2, 5, 6, 15]
})

res_12 = resumen_categoria(df_cat)
row_tech = res_12[res_12["categoria"] == "Tech"]
assert row_tech["total_ventas"].values[0] == 400.0
assert row_tech["promedio_unidades"].values[0] == 4.0
print("✅ Test Ejercicio 1.2 superado.")

#### Ejercicio 1.3: Cruce de Datos (Left Join) (CE1.d)
Implementa `fusionar_clientes_pedidos(df_clientes, df_pedidos)` que:
1. Haga un `left join` sobre `cliente_id`.
2. Rellene con `0.0` los nulos de la columna `monto`.

In [ ]:
def fusionar_clientes_pedidos(df_clientes: pd.DataFrame, df_pedidos: pd.DataFrame) -> pd.DataFrame:
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa la función fusionar_clientes_pedidos")

In [ ]:
# TEST 1.3
df_c = pd.DataFrame({"cliente_id": [1, 2, 3], "nombre": ["Ana", "Juan", "Pedro"]})
df_p = pd.DataFrame({"cliente_id": [1, 2], "monto": [100.0, 250.0]})

res_13 = fusionar_clientes_pedidos(df_c, df_p)
assert len(res_13) == 3
assert res_13.loc[res_13["cliente_id"] == 3, "monto"].values[0] == 0.0
print("✅ Test Ejercicio 1.3 superado.")